In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import GRU, Dense, Flatten,Input, Dropout, Conv2D, MaxPooling2D, Bidirectional, LSTM, Dense
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from tensorflow.keras.regularizers import l2
from tensorflow.keras import Model
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
from keras.models import load_model

In [ ]:
# col_remove = ['SIP', 'DIP', 'SPort', 'DPort', 'SIT', 'STT']
# fl_mu_data=fl_mu_data.drop(col_remove, axis=1, inplace=False)
# irr_list = []
# for i in fl_mu_data:
#     if len(fl_mu_data[i].unique())<2:
#         irr_list.append(i)
# fl_mu_data=fl_mu_data.drop(irr_list, axis=1, inplace=False)

Global Server Space

In [5]:
#Binary classification model develop by the global server
inputs = Input(shape=(1, 73), name='input_layer')

# GRU layer with 64 units and return sequences set to False
gru_layer1 = GRU(64, name='gru_layer1', kernel_regularizer=l2(0.01), return_sequences=True)(inputs)

# Dropout layer for regularization
dropout_layer = Dropout(0.2, name='dropout_layer')(gru_layer1)

gru_layer2 = GRU(32, name='gru_layer2', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer)

# Dropout layer for regularization
dropout_layer2= Dropout(0.2, name='dropout_layer2')(gru_layer2)

gru_layer3 = GRU(16, name='gru_layer3', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer2)

# Dropout layer for regularization
dropout_layer3= Dropout(0.2, name='dropout_layer3')(gru_layer3)

flatten_layer = Flatten(name='flatten_layer')(dropout_layer3)
# Dense layer with 1 unit (binary classification) or more (multi-class classification)
output_layer = Dense(2, activation='sigmoid', name='output_layer',kernel_regularizer=l2(0.01))(flatten_layer)

# Build the model
model_bin = Model(inputs=inputs, outputs=output_layer, name='GRU_Model')

# Compile the model
model_bin.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Model summary to see the structure
model_bin.summary()



Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 73)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer1 (GRU)                │ (None, 1, 64)          │        26,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer2 (GRU)                │ (None, 1, 32)          │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer2 (Dropout)        │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer3 (GRU)                │ (None, 1, 16)          │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer3 (Dropout)        │ (None, 1, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,530 (150.51 KB)

 Trainable params: 38,530 (150.51 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/type/client1_felids_dnn_type_2nd_FL_weight.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/type/client2_felids_dnn_type_2nd_FL_weight.keras")
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/type/client3_felids_dnn_type_2nd_FL_weight.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [ ]:
# felids_dnn_cate.save(r'D:\linthoi\FL_models\new\glob_models\Category_glob_model\firstinitialized\felids_dnn_cate.h5')

In [7]:

felids_cnn_cate = load_model(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\felids_dnn_type.h5')
felids_cnn_cate.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
felids_cnn_cate.set_weights(avg_weights)
felids_cnn_cate.save('D:/linthoi/FL_models/new/glob_models/Type_glob_model/updated_models/3rd_round_felids_dnn_type_glob_model.h5')

In [3]:
avg_weights2=avg_weights
avg_weights2

[<tf.Tensor: shape=(73, 192), dtype=float32, numpy=
 array([[-1.5453117e-05, -4.3941430e-08,  2.8448042e-06, ...,
         -7.5442070e-04,  6.6250082e-06, -1.5616279e-06],
        [-1.4861105e-07, -7.6695706e-08, -5.2906825e-07, ...,
         -8.3497604e-03,  3.9986390e-04, -5.4779152e-06],
        [-6.6401403e-06, -3.4456797e-08, -4.3640571e-06, ...,
         -8.3497595e-03,  3.9986390e-04, -5.4779139e-06],
        ...,
        [-9.0513652e-08, -2.2985716e-06,  2.9008447e-08, ...,
          9.7725412e-04, -5.6145374e-05,  6.1549144e-07],
        [ 6.0273436e-08,  7.9550074e-07, -1.0749691e-07, ...,
          2.3593330e-04, -4.5955098e-06,  3.4607410e-09],
        [ 1.2780851e-08,  4.9269357e-09,  4.5599506e-08, ...,
          9.1888197e-04, -1.7376538e-05,  7.9462303e-08]], dtype=float32)>,
 <tf.Tensor: shape=(64, 192), dtype=float32, numpy=
 array([[ 0.06120588, -0.05009326,  0.05515286, ...,  0.0048919 ,
         -0.06567136,  0.04619128],
        [-0.00588059,  0.02698278, -0.04467

In [3]:
avg_weights

[<tf.Tensor: shape=(73, 192), dtype=float32, numpy=
 array([[ 2.0615380e-05,  2.1963764e-05, -9.3903774e-11, ...,
         -4.7245881e-04,  2.1522026e-06, -1.2297706e-07],
        [-1.8469520e-05, -1.5408998e-05, -1.1696947e-06, ...,
          6.2139151e-03,  2.5502632e-06, -1.3889918e-06],
        [-1.9360969e-07, -1.5402475e-05,  7.3042082e-07, ...,
          6.2139160e-03,  2.5502611e-06, -1.3889918e-06],
        ...,
        [ 6.3940270e-08, -8.2182873e-07,  1.6790413e-08, ...,
         -1.3486906e-04, -6.6370791e-07,  4.8949448e-07],
        [ 4.6779918e-09, -5.2856326e-06,  2.7018895e-10, ...,
         -2.8130968e-05,  3.6639264e-07,  1.3819619e-07],
        [ 1.4966217e-06, -1.4396206e-05,  9.8140144e-06, ...,
          2.4264505e-04, -3.2332818e-07,  1.7836901e-07]], dtype=float32)>,
 <tf.Tensor: shape=(64, 192), dtype=float32, numpy=
 array([[ 0.05188044, -0.0524786 ,  0.03441697, ...,  0.03769286,
         -0.07567439,  0.03856959],
        [ 0.00256385,  0.05221403, -0.02436

In [6]:
model_bin.set_weights(avg_weights2)
model_bin.save('D:/linthoi/FL_models/new/2ndbin_glob_model.h5')

In [9]:
mod = load_model('D:/linthoi/FL_models/new/1stbin_glob_model.h5')

In [10]:
mod.summary()

Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 73)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer1 (GRU)                │ (None, 1, 64)          │        26,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer2 (GRU)                │ (None, 1, 32)          │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer2 (Dropout)        │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer3 (GRU)                │ (None, 1, 16)          │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer3 (Dropout)        │ (None, 1, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,532 (150.52 KB)

 Trainable params: 38,530 (150.51 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

Multiclass classification of 4 class (category)

In [4]:
#Binary classification model develop by the global server
inputs = Input(shape=(1, 72), name='input_layer')

# GRU layer with 64 units and return sequences set to False
gru_layer1 = GRU(64, name='gru_layer1', kernel_regularizer=l2(0.01), return_sequences=True)(inputs)

# Dropout layer for regularization
dropout_layer = Dropout(0.2, name='dropout_layer')(gru_layer1)

gru_layer2 = GRU(32, name='gru_layer2', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer)

# Dropout layer for regularization
dropout_layer2= Dropout(0.2, name='dropout_layer2')(gru_layer2)

gru_layer3 = GRU(16, name='gru_layer3', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer2)

# Dropout layer for regularization
dropout_layer3= Dropout(0.2, name='dropout_layer3')(gru_layer3)

flatten_layer = Flatten(name='flatten_layer')(dropout_layer3)
# Dense layer with 1 unit (binary classification) or more (multi-class classification)
output_layer = Dense(4, activation='sigmoid', name='output_layer',kernel_regularizer=l2(0.01))(flatten_layer)

# Build the model
model_cate = Model(inputs=inputs, outputs=output_layer, name='GRU_Model')

# Compile the model
model_cate.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Model summary to see the structure
model_cate.summary()



Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 72)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer1 (GRU)                │ (None, 1, 64)          │        26,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer2 (GRU)                │ (None, 1, 32)          │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer2 (Dropout)        │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer3 (GRU)                │ (None, 1, 16)          │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer3 (Dropout)        │ (None, 1, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 4)              │            68 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,372 (149.89 KB)

 Trainable params: 38,372 (149.89 KB)

 Non-trainable params: 0 (0.00 B)

5 class multiclass classification (type)

In [ ]:
#Binary classification model develop by the global server
inputs = Input(shape=(1, 72), name='input_layer')

# GRU layer with 64 units and return sequences set to False
gru_layer1 = GRU(64, name='gru_layer1', kernel_regularizer=l2(0.01), return_sequences=True)(inputs)

# Dropout layer for regularization
dropout_layer = Dropout(0.2, name='dropout_layer')(gru_layer1)

gru_layer2 = GRU(32, name='gru_layer2', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer)

# Dropout layer for regularization
dropout_layer2= Dropout(0.2, name='dropout_layer2')(gru_layer2)

gru_layer3 = GRU(16, name='gru_layer3', kernel_regularizer=l2(0.01), return_sequences=True)(dropout_layer2)

# Dropout layer for regularization
dropout_layer3= Dropout(0.2, name='dropout_layer3')(gru_layer3)

flatten_layer = Flatten(name='flatten_layer')(dropout_layer3)
# Dense layer with 1 unit (binary classification) or more (multi-class classification)
output_layer = Dense(5, activation='softmax', name='output_layer',kernel_regularizer=l2(0.01))(flatten_layer)

# Build the model
model_type = Model(inputs=inputs, outputs=output_layer, name='GRU_Model')

# Compile the model
model_type.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Model summary to see the structure
model_type.summary()



Model: "GRU_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 72)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer1 (GRU)                │ (None, 1, 64)          │        26,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer2 (GRU)                │ (None, 1, 32)          │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer2 (Dropout)        │ (None, 1, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer3 (GRU)                │ (None, 1, 16)          │         2,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer3 (Dropout)        │ (None, 1, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_layer (Flatten)         │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 5)              │            85 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,389 (149.96 KB)

 Trainable params: 38,389 (149.96 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
model_type.save('D:/linthoi/FL_models/new/glob_models/Type_glob_model/firstinitialized/Type_glob_model.h5')

In [6]:
model_cate.save('D:/linthoi/FL_models/new/Cate_glob_model.h5')

In [4]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/category/client1_FELIDS_dnn_bfore_FL_weight.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/category/client2_FELIDS_dnn_bfore_FL_weight.keras")
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/category/client3_FELIDS_dnn_bfore_FL_weight.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [14]:
avg_weights

[<tf.Tensor: shape=(72, 192), dtype=float32, numpy=
 array([[-2.0249185e-05,  3.6872987e-06, -2.3397332e-07, ...,
          7.7611148e-05, -4.2032748e-02, -8.1352238e-04],
        [-1.2414173e-05,  3.6594665e-06, -8.2189501e-08, ...,
          7.7610988e-05, -4.2032737e-02, -8.1352220e-04],
        [ 2.2464408e-05, -4.5244306e-06,  7.9363718e-07, ...,
         -1.3194076e-03, -9.8875421e-04,  3.4022224e-04],
        ...,
        [ 9.1654920e-06, -7.2735389e-07,  6.3535372e-06, ...,
          2.1519612e-04, -1.9961628e-03, -8.3207391e-04],
        [ 2.7053766e-06, -2.6887579e-07, -1.8464969e-07, ...,
         -2.8611365e-04,  2.4209054e-04,  7.6486169e-05],
        [ 3.3286942e-06,  1.5817396e-06,  1.4913220e-06, ...,
         -2.5989732e-04,  2.4677085e-04,  7.4844451e-05]], dtype=float32)>,
 <tf.Tensor: shape=(64, 192), dtype=float32, numpy=
 array([[ 0.00623417,  0.06134047,  0.04429818, ...,  0.04977449,
         -0.09665594, -0.00318419],
        [ 0.04135112, -0.02202415, -0.05410

In [5]:
avg_weight

<tf.Tensor: shape=(5,), dtype=float32, numpy=
array([-3.6188767, -1.7451539, -1.2351677, -4.5504284, -1.6487325],
      dtype=float32)>

In [16]:
# model_updte_type = load_model('D:/linthoi/FL_models/new/glob_models/Type_glob_model/updated_models/2nd_round_type_glob_model.h5')
# w= model_updte_type.get_weights()
# w

In [5]:

model_updte_type = load_model(r'D:\linthoi\FL_models\new\glob_models\Category_glob_model\firstinitialized\felids_dnn_cate.h5')
model_updte_type.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_updte_type.set_weights(avg_weights)
model_updte_type.save('D:/linthoi/FL_models/new/glob_models/Category_glob_model/updated_models/1st_round_felids_dnn_cate_glob_model.h5')

CNN_bilstm and dnn from paper (https://www.sciencedirect.com/science/article/pii/S1570870524001513?ref=pdf_download&fr=RR-2&rr=92d004ee38dd9371)

In [7]:
#FIRST CNN BILSTM MODEL
#input_layer = Input(shape=(height, width, channels), name='input_layer')
input_layer = Input(shape=(72, 1), name="input_layer")

# Convolutional Layer 1: 32 filters of size (3, 3), ReLU activation, named 'conv_1'
layer1 = Conv2D(32, kernel_size=(3, 3), activation='relu', name='conv_1')(input_layer)

# Max Pooling Layer: Pool size of (2, 2), named 'maxpool_1'
layer2 = MaxPooling2D(pool_size=(2, 2), name='maxpool_1')(layer1)

# Dropout Layer after the Max Pooling Layer (with 0.2 rate)
layer2_dropout = Dropout(0.2, name='dropout_1')(layer2)

# Recurrent Layer 1: Bidirectional LSTM with 32 units, named 'bilstm_1'
layer3 = Bidirectional(LSTM(32, return_sequences=True), name='bilstm_1')(layer2_dropout)

# Dropout Layer after the Bidirectional LSTM (with 0.2 rate)
layer3_dropout = Dropout(0.2, name='dropout_2')(layer3)

# Recurrent Layer 2: Bidirectional LSTM with 16 units, named 'bilstm_2'
layer4 = Bidirectional(LSTM(16), name='bilstm_2')(layer3_dropout)

# Dense Layer: 1 unit with Sigmoid activation for binary classification, named 'dense_1'
output_layer = Dense(1, activation='sigmoid', name='dense_1')(layer4)

# Create the model using the functional API
model_1 = Model(inputs=input_layer, outputs=output_layer, name='model_1')

# Compile the model with Binary Crossentropy loss and Adam optimizer
model_1.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Summary of the model architecture
model_1.summary()

ValueError: Input 0 of layer "conv_1" is incompatible with the layer: expected min_ndim=4, found ndim=3. Full shape received: (None, 72, 1)

WAIT FOR A WHILE

DNN from (https://www.sciencedirect.com/science/article/pii/S0743731522000570)

In [32]:
import random

In [34]:
hidden_nodes = random.randint(64, 80)  # Hidden nodes between 64 and 80
hidden_layers = random.randint(1, 2)  # Hidden layers between 1 and 2
dropout_rate = round(random.uniform(0.1, 0.4), 2)  # Dropout between 0.1 and 0.4

# Define the input layer (assuming 72 timesteps and 1 feature per timestep)
input_layer = Input(shape=(72, 1), name="input_layer")  # Input shape (72, 1) for sequential data

# Layer 1 - First Hidden Layer
layer1 = Dense(hidden_nodes, activation='relu')(input_layer)

# Layer 2 - Dropout Layer after the first hidden layer
layer2 = Dropout(dropout_rate)(layer1)

# Layer 3 - Second Hidden Layer (if there are 2 hidden layers)
if hidden_layers > 1:
    layer3 = Dense(hidden_nodes, activation='relu')(layer2)
    
    # Layer 4 - Dropout after the second hidden layer
    layer4 = Dropout(dropout_rate)(layer3)
else:
    layer4 = layer2  # If there's only one hidden layer, just use the dropout layer from the first layer

# Flatten the output to feed into the final dense layer
flattened_output = Flatten()(layer4)

# Layer 5 - Output Layer (10 classes)
output_layer = Dense(4, activation='softmax')(flattened_output)  # 10 output classes, adjust as needed

# Create the model
felids_dnn_cate = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
felids_dnn_cate.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
felids_dnn_cate.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 72, 73)         │           146 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 72, 73)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 72, 73)         │         5,402 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 72, 73)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 5256)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4)              │        21,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,576 (103.81 KB)

 Trainable params: 26,576 (103.81 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
felids_dnn_cate.save(r'D:\linthoi\FL_models\new\glob_models\Category_glob_model\firstinitialized\felids_dnn_cate.h5')

In [2]:
import random

In [4]:
hidden_nodes = random.randint(64, 80)  # Hidden nodes between 64 and 80
hidden_layers = random.randint(1, 2)  # Hidden layers between 1 and 2
dropout_rate = round(random.uniform(0.1, 0.4), 2)  # Dropout between 0.1 and 0.4

# Define the input layer (assuming 72 timesteps and 1 feature per timestep)
input_layer = Input(shape=(72, 1), name="input_layer")  # Input shape (72, 1) for sequential data

# Layer 1 - First Hidden Layer
layer1 = Dense(hidden_nodes, activation='relu')(input_layer)

# Layer 2 - Dropout Layer after the first hidden layer
layer2 = Dropout(dropout_rate)(layer1)

# Layer 3 - Second Hidden Layer (if there are 2 hidden layers)
if hidden_layers > 1:
    layer3 = Dense(hidden_nodes, activation='relu')(layer2)
    
    # Layer 4 - Dropout after the second hidden layer
    layer4 = Dropout(dropout_rate)(layer3)
else:
    layer4 = layer2  # If there's only one hidden layer, just use the dropout layer from the first layer

# Flatten the output to feed into the final dense layer
flattened_output = Flatten()(layer4)

# Layer 5 - Output Layer (10 classes)
output_layer = Dense(2, activation='sigmoid')(flattened_output)  # 10 output classes, adjust as needed

# Create the model
felids_dnn_bin = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
felids_dnn_bin.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
felids_dnn_bin.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 72, 67)         │           134 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 72, 67)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4824)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 2)              │         9,650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,784 (38.22 KB)

 Trainable params: 9,784 (38.22 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
felids_dnn_bin.save(r'D:\linthoi\FL_models\new\glob_models\Binary_glob_model\firstinitialized\felids_dnn_bin.h5')

In [8]:
hidden_nodes = random.randint(64, 80)  # Hidden nodes between 64 and 80
hidden_layers = random.randint(1, 2)  # Hidden layers between 1 and 2
dropout_rate = round(random.uniform(0.1, 0.4), 2)  # Dropout between 0.1 and 0.4

# Define the input layer (assuming 72 timesteps and 1 feature per timestep)
input_layer = Input(shape=(72, 1), name="input_layer")  # Input shape (72, 1) for sequential data

# Layer 1 - First Hidden Layer
layer1 = Dense(hidden_nodes, activation='relu')(input_layer)

# Layer 2 - Dropout Layer after the first hidden layer
layer2 = Dropout(dropout_rate)(layer1)

# Layer 3 - Second Hidden Layer (if there are 2 hidden layers)
if hidden_layers > 1:
    layer3 = Dense(hidden_nodes, activation='relu')(layer2)
    
    # Layer 4 - Dropout after the second hidden layer
    layer4 = Dropout(dropout_rate)(layer3)
else:
    layer4 = layer2  # If there's only one hidden layer, just use the dropout layer from the first layer

# Flatten the output to feed into the final dense layer
flattened_output = Flatten()(layer4)

# Layer 5 - Output Layer (10 classes)
output_layer = Dense(5, activation='softmax')(flattened_output)  # 10 output classes, adjust as needed

# Create the model
felids_dnn_type = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
felids_dnn_type.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
felids_dnn_type.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 72, 71)         │           142 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 72, 71)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5112)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │        25,565 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,707 (100.42 KB)

 Trainable params: 25,707 (100.42 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
felids_dnn_type.save(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\felids_dnn_type.h5')

Felids RNN model

In [7]:
input_layer = Input(shape=(72, 1), name="input_layer")  # Input shape (30, 1) for sequential data

# Layer 1 - First LSTM Layer
layer1 = LSTM(units=20, return_sequences=True)(input_layer)

# Layer 2 - Dropout Layer after the first LSTM layer
layer2 = Dropout(0.2)(layer1)

# Layer 3 - Second LSTM Layer
layer3 = LSTM(units=80, return_sequences=True)(layer2)

# Layer 4 - Dropout Layer after the second LSTM layer
layer4 = Dropout(0.2)(layer3)

# Layer 5 - Third LSTM Layer
layer5 = LSTM(units=80)(layer4)

# Layer 6 - Dropout Layer after the third LSTM layer
layer6 = Dropout(0.2)(layer5)

# Output Layer
output_layer = Dense(2,activation='sigmoid')(layer6)  

# Create the model
model_RNN_bin = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
model_RNN_bin.compile(optimizer='adam', loss='binary_crossentropy',  metrics=['accuracy'])  # Change loss for different tasks

# Print model summary
model_RNN_bin.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 72, 20)         │         1,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 72, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 72, 80)         │        32,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 72, 80)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 80)             │        51,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           162 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,762 (335.01 KB)

 Trainable params: 85,762 (335.01 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model_RNN_bin.save(r'D:\linthoi\FL_models\new\glob_models\Binary_glob_model\firstinitialized\model_RNN_bin.h5')

In [9]:
input_layer = Input(shape=(72, 1), name="input_layer")  # Input shape (30, 1) for sequential data

# Layer 1 - First LSTM Layer
layer1 = LSTM(units=20, return_sequences=True)(input_layer)

# Layer 2 - Dropout Layer after the first LSTM layer
layer2 = Dropout(0.2)(layer1)

# Layer 3 - Second LSTM Layer
layer3 = LSTM(units=80, return_sequences=True)(layer2)

# Layer 4 - Dropout Layer after the second LSTM layer
layer4 = Dropout(0.2)(layer3)

# Layer 5 - Third LSTM Layer
layer5 = LSTM(units=80)(layer4)

# Layer 6 - Dropout Layer after the third LSTM layer
layer6 = Dropout(0.2)(layer5)

# Output Layer
output_layer = Dense(5,activation='softmax')(layer6)  

# Create the model
model_RNN_cate = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
model_RNN_cate.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])  # Change loss for different tasks

# Print model summary
model_RNN_cate.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 72, 20)         │         1,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 72, 20)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 72, 80)         │        32,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 72, 80)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_8 (LSTM)                   │ (None, 80)             │        51,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 80)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           405 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 86,005 (335.96 KB)

 Trainable params: 86,005 (335.96 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model_RNN_cate.save(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\felids_RNN_type.h5')

FELIDS CNN

In [10]:
from tensorflow.keras import layers, models

In [11]:
# Create a Sequential model
model = models.Sequential()

# Add 3 convolutional layers (or 2, depending on your choice)
model.add(layers.Conv1D(32, 3, activation='relu', input_shape=(72, 1)))  # Input shape: (72, 1)
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(layers.Conv1D(128, 3, activation='relu'))

# Add global average pooling
model.add(layers.GlobalAveragePooling1D())

# Add hidden dense layers
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.3))  # Example dropout rate
model.add(layers.Dense(120, activation='relu'))  # Second dense layer
model.add(layers.Dropout(0.2))  # Example dropout rate
model.add(layers.Dense(120, activation='relu'))  # Optional third dense layer

# Output layer
model.add(layers.Dense(2, activation='sigmoid'))  # Assuming binary classification

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summarize the model
model.summary()

c:\Users\ranja\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 70, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 68, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 66, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 120)            │        15,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 120)            │        14,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │           242 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,794 (303.88 KB)

 Trainable params: 77,794 (303.88 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
model.save(r'D:\linthoi\FL_models\new\glob_models\Binary_glob_model\firstinitialized\felids_cnn_bin.h5')

In [10]:
# Create a Sequential model
model = models.Sequential()

# Add 3 convolutional layers (or 2, depending on your choice)
model.add(layers.Conv1D(32, 3, activation='relu', input_shape=(72, 1)))  # Input shape: (72, 1)
model.add(layers.Conv1D(64, 3, activation='relu'))
model.add(layers.Conv1D(128, 3, activation='relu'))

# Add global average pooling
model.add(layers.GlobalAveragePooling1D())

# Add hidden dense layers
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.3))  # Example dropout rate
model.add(layers.Dense(120, activation='relu'))  # Second dense layer
model.add(layers.Dropout(0.2))  # Example dropout rate
model.add(layers.Dense(120, activation='relu'))  # Optional third dense layer

# Output layer
model.add(layers.Dense(5, activation='softmax'))  # Assuming binary classification

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summarize the model
model.summary()

c:\Users\ranja\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 70, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 68, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 66, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 120)            │        15,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 120)            │        14,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           605 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 78,157 (305.30 KB)

 Trainable params: 78,157 (305.30 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.save(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\felids_cnn_type_glob_model.h5')

PCC_CNN from paper (https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=10495036)

In [55]:
pcc_cnn = models.Sequential()

# Input Layer (input_shape as first layer)
input_layer = layers.Input(shape=(72, 1), name="input_layer")

# 1st Convolutional Layer (91x64) with ReLU activation
pcc_cnn1 = layers.Conv1D(filters=64, kernel_size=3, activation='relu', name="conv1")(input_layer)

# 2nd Convolutional Layer (10x64) with ReLU activation
pcc_cnn2 = layers.Conv1D(filters=64, kernel_size=10, activation='relu', name="conv2")(pcc_cnn1)

# MaxPooling1D Layer
pcc_cnn3 = layers.MaxPooling1D(pool_size=2, name="maxpool1")(pcc_cnn2)

# Flatten the output from the previous layer
pcc_cnn4 = layers.Flatten(name="flatten")(pcc_cnn3)

# Dense Layer with 128 units and ReLU activation
pcc_cnn5 = layers.Dense(128, activation='relu', name="dense1")(pcc_cnn4)

# Dropout Layer with rate of 30%
pcc_cnn6 = layers.Dropout(0.3, name="dropout")(pcc_cnn5)

# Output Dense Layer with 2 units (for binary classification) using Softmax
output_pcc_cnn = layers.Dense(2, activation='softmax', name="dense_output")(pcc_cnn6)

# Create the model using the input and output layers
pcc_cnn_bin = models.Model(inputs=input_layer, outputs=output_pcc_cnn)

# Compile the model using Adam optimizer
pcc_cnn_bin.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
pcc_cnn_bin.summary()


Model: "functional_182"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 70, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 61, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool1 (MaxPooling1D)         │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       245,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_output (Dense)            │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 287,426 (1.10 MB)

 Trainable params: 287,426 (1.10 MB)

 Non-trainable params: 0 (0.00 B)

In [56]:
pcc_cnn = models.Sequential()

# Input Layer (input_shape as first layer)
input_layer = layers.Input(shape=(72, 1), name="input_layer")

# 1st Convolutional Layer (91x64) with ReLU activation
pcc_cnn1 = layers.Conv1D(filters=64, kernel_size=3, activation='relu', name="conv1")(input_layer)

# 2nd Convolutional Layer (10x64) with ReLU activation
pcc_cnn2 = layers.Conv1D(filters=64, kernel_size=10, activation='relu', name="conv2")(pcc_cnn1)

# MaxPooling1D Layer
pcc_cnn3 = layers.MaxPooling1D(pool_size=2, name="maxpool1")(pcc_cnn2)

# Flatten the output from the previous layer
pcc_cnn4 = layers.Flatten(name="flatten")(pcc_cnn3)

# Dense Layer with 128 units and ReLU activation
pcc_cnn5 = layers.Dense(128, activation='relu', name="dense1")(pcc_cnn4)

# Dropout Layer with rate of 30%
pcc_cnn6 = layers.Dropout(0.3, name="dropout")(pcc_cnn5)

# Output Dense Layer with 2 units (for binary classification) using Softmax
output_pcc_cnn = layers.Dense(4, activation='softmax', name="dense_output")(pcc_cnn6)

# Create the model using the input and output layers
pcc_cnn_cate = models.Model(inputs=input_layer, outputs=output_pcc_cnn)

# Compile the model using Adam optimizer
pcc_cnn_cate.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
pcc_cnn_cate.summary()


Model: "functional_183"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 70, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 61, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool1 (MaxPooling1D)         │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       245,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_output (Dense)            │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 287,684 (1.10 MB)

 Trainable params: 287,684 (1.10 MB)

 Non-trainable params: 0 (0.00 B)

In [59]:
pcc_cnn = models.Sequential()

# Input Layer (input_shape as first layer)
input_layer = layers.Input(shape=(72, 1), name="input_layer")

# 1st Convolutional Layer (91x64) with ReLU activation
pcc_cnn1 = layers.Conv1D(filters=64, kernel_size=3, activation='relu', name="conv1")(input_layer)

# 2nd Convolutional Layer (10x64) with ReLU activation
pcc_cnn2 = layers.Conv1D(filters=64, kernel_size=10, activation='relu', name="conv2")(pcc_cnn1)

# MaxPooling1D Layer
pcc_cnn3 = layers.MaxPooling1D(pool_size=2, name="maxpool1")(pcc_cnn2)

# Flatten the output from the previous layer
pcc_cnn4 = layers.Flatten(name="flatten")(pcc_cnn3)

# Dense Layer with 128 units and ReLU activation
pcc_cnn5 = layers.Dense(128, activation='relu', name="dense1")(pcc_cnn4)

# Dropout Layer with rate of 30%
pcc_cnn6 = layers.Dropout(0.3, name="dropout")(pcc_cnn5)

# Output Dense Layer with 2 units (for binary classification) using Softmax
output_pcc_cnn = layers.Dense(5, activation='softmax', name="dense_output")(pcc_cnn6)

# Create the model using the input and output layers
pcc_cnn_type = models.Model(inputs=input_layer, outputs=output_pcc_cnn)

# Compile the model using Adam optimizer
pcc_cnn_type.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
pcc_cnn_type.summary()


Model: "functional_184"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 72, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv1D)                  │ (None, 70, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 61, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool1 (MaxPooling1D)         │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 128)            │       245,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_output (Dense)            │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 287,813 (1.10 MB)

 Trainable params: 287,813 (1.10 MB)

 Non-trainable params: 0 (0.00 B)

In [60]:
pcc_cnn_type.save(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\pcc_cnn_type_glob_model.h5')

trial from a paper of cnn-gru

In [32]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dropout, Dense
from tensorflow.keras import layers, models

In [ ]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dropout, Dense
from tensorflow.keras.models import Sequential
model = Sequential()
model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(None, 72)))
model.add(MaxPooling1D(pool_size=2))
model.add(GRU(units=64, return_sequences=False))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

c:\Users\ranja\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, None, 64)       │        13,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, None, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,913 (152.00 KB)

 Trainable params: 38,913 (152.00 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
# model_updte_cate.get_weights()

Aggregation of categorical model.

In [2]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/type/client1_FELIDS_rnn_type_2nd_FL_weight.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/type/client2_FELIDS_rnn_type_2nd_FL_weight.keras")
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/type/client3_FELIDS_rnn_type_2nd_FL_weight.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [3]:

model_updte_cate = load_model(r'D:\linthoi\FL_models\new\glob_models\Type_glob_model\firstinitialized\felids_RNN_type.h5')
model_updte_cate.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_updte_cate.set_weights(avg_weights)
model_updte_cate.save('D:/linthoi/FL_models/new/glob_models/Type_glob_model/updated_models/3rd_round_felids_rnn_type_glob_model.h5')

In [4]:
felids_RNN_type.summary()

NameError: name 'felids_RNN_type' is not defined

In [9]:
model_updte_cate.get_weights()

[array([[ 3.6570475e-06,  9.1055199e-06,  2.0592961e-07, ...,
          3.6932111e-02,  3.7287773e-06,  4.0283449e-02],
        [ 4.7458429e-06,  2.8273411e-05,  7.1889207e-07, ...,
          3.6932170e-02,  2.0626223e-06,  4.0285159e-02],
        [ 3.0371979e-05, -2.5268928e-05, -3.5612466e-06, ...,
          2.1372831e-03, -5.1577058e-06,  1.9857076e-03],
        ...,
        [ 1.3532869e-05,  1.0200790e-05, -4.8873458e-06, ...,
         -6.5766000e-03, -9.2523123e-06, -7.6982309e-04],
        [ 1.0435142e-05,  8.1477782e-07, -3.9043573e-07, ...,
         -2.0116873e-03, -3.4725235e-06, -1.8328846e-03],
        [ 1.9753555e-08, -7.8738680e-07,  1.4062063e-06, ...,
         -1.8049650e-03, -1.8049468e-06, -1.6697253e-03]], dtype=float32),
 array([[ 0.02949429,  0.09929314, -0.10583674, ...,  0.01996378,
         -0.01320487,  0.05570148],
        [-0.07596169,  0.08029008, -0.07850148, ...,  0.11750399,
          0.03812586,  0.02100072],
        [-0.00711831,  0.09585825,  0.11256683

CNN_GRU from MDPI

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GRU, Dense, Dropout, Flatten

In [7]:


# Define the model
model = Sequential()

# Step 1: 1D Convolution Layer
# Ensure input sequence length is greater than or equal to the kernel size
model.add(Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(1, 72), name="conv1d_layer"))

# Step 2: 1D MaxPooling Layer
model.add(MaxPooling1D(pool_size=1, name="maxpooling_layer"))

# Step 3: Fully Connected (Dense) Layer
model.add(Dense(128, activation='relu', name="fully_connected_layer"))

# Step 4: GRU Layer
model.add(GRU(units=128, activation='tanh', return_sequences=False, name="gru_layer"))

# Step 5: Dropout Layer to prevent overfitting
model.add(Dropout(0.5, name="dropout_layer"))

# Step 6: Output Layer for Multiclass Classification
model.add(Dense(2, activation='softmax', name="output_layer"))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary to check the layers
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_layer (Conv1D)           │ (None, 1, 64)          │         4,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling_layer (MaxPooling1D) │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fully_connected_layer (Dense)   │ (None, 1, 128)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer (GRU)                 │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,322 (438.76 KB)

 Trainable params: 112,322 (438.76 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:


# Define the model
model = Sequential()

# Step 1: 1D Convolution Layer
# Ensure input sequence length is greater than or equal to the kernel size
model.add(Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(1, 72), name="conv1d_layer"))

# Step 2: 1D MaxPooling Layer
model.add(MaxPooling1D(pool_size=1, name="maxpooling_layer"))

# Step 3: Fully Connected (Dense) Layer
model.add(Dense(128, activation='relu', name="fully_connected_layer"))

# Step 4: GRU Layer
model.add(GRU(units=128, activation='tanh', return_sequences=False, name="gru_layer"))

# Step 5: Dropout Layer to prevent overfitting
model.add(Dropout(0.5, name="dropout_layer"))

# Step 6: Output Layer for Multiclass Classification
model.add(Dense(7, activation='softmax', name="output_layer"))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary to check the layers
model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_layer (Conv1D)           │ (None, 1, 64)          │         4,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling_layer (MaxPooling1D) │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fully_connected_layer (Dense)   │ (None, 1, 128)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer (GRU)                 │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,967 (441.28 KB)

 Trainable params: 112,967 (441.28 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.save('D:/linthoi/FL_models/new/glob_models/Category_glob_model/first_initialized_model/CNN_GRU_MDPI_cate_glob_model.h5')

In [ ]:


# Define the model
model = Sequential()

# Step 1: 1D Convolution Layer
# Ensure input sequence length is greater than or equal to the kernel size
model.add(Conv1D(filters=64, kernel_size=1, activation='relu', input_shape=(1, 72), name="conv1d_layer"))

# Step 2: 1D MaxPooling Layer
model.add(MaxPooling1D(pool_size=1, name="maxpooling_layer"))

# Step 3: Fully Connected (Dense) Layer
model.add(Dense(128, activation='relu', name="fully_connected_layer"))

# Step 4: GRU Layer
model.add(GRU(units=128, activation='tanh', return_sequences=False, name="gru_layer"))

# Step 5: Dropout Layer to prevent overfitting
model.add(Dropout(0.5, name="dropout_layer"))

# Step 6: Output Layer for Multiclass Classification
model.add(Dense(5, activation='softmax', name="output_layer"))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print the model summary to check the layers
model.summary()


c:\Users\ranja\miniconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_layer (Conv1D)           │ (None, 1, 64)          │         4,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling_layer (MaxPooling1D) │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fully_connected_layer (Dense)   │ (None, 1, 128)         │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_layer (GRU)                 │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_layer (Dropout)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,709 (440.27 KB)

 Trainable params: 112,709 (440.27 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.save('D:/linthoi/FL_models/new/glob_models/Type_glob_model/firstinitialized/CNN_GRU_MDPI_Type_glob_model.h5')

Model aggregation point

In [21]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/category/client1_cnn_gru_mdpi_cate_bfore_FL_weight_iid.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/category/client2_cnn_gru_mdpi_cate_bfore_FL_weight_iid.keras")
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/category/client3_cnn_gru_mdpi_cate_beforeFL_weight_iid.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [22]:

cnn_gru_mdpi = load_model('D:/linthoi/FL_models/new/glob_models/Category_glob_model/first_initialized_model/CNN_GRU_MDPI_cate_glob_model.h5')
cnn_gru_mdpi.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_gru_mdpi.set_weights(avg_weights)
cnn_gru_mdpi.save('D:/linthoi/FL_models/new/glob_models/Category_glob_model/updated_models/1stround_CNN_GRU_MDPI_cate_model.h5')

final_model_aggregation

In [26]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/binary/client1_cnn_gru_mdpi_bin_bfore_FL_weight.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/binary/client2_cnn_gru_mdpi_binary_bfore_FL_weight.keras") 
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/binary/client3_cnn_gru_mdpi_binary_beforeFL_weight.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [28]:

cnn_gru_mdpi = load_model('D:/linthoi/FL_models/new/glob_models/Binary_glob_model/firstinitialized/CNN_GRU_MDPI_binary_glob_model.h5')
cnn_gru_mdpi.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_gru_mdpi.set_weights(avg_weights)
cnn_gru_mdpi.save('D:/linthoi/FL_models/new/glob_models/Binary_glob_model/updated_models/Final_CNN_GRU_MDPI_binary_model.h5')

In [18]:
load_model_C1 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client1/type/client1_cnn_gru_mdpi_type_3rdround_FL_weight.keras")
WC1=load_model_C1.get_weights()
load_model_C2 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client2/type/client2_cnn_gru_mdpi_type_3rdround_FL_weight.keras")
WC2=load_model_C2.get_weights()
load_model_C3 = tf.keras.models.load_model("D:/linthoi/FL_models/new/client3/type/client3_cnn_gru_mdpi_type_3rdround_FL_weight.keras")
WC3=load_model_C3.get_weights()
weights = [WC1, WC2, WC3]
avg_weights = []

for weights_list_tuple in zip(*weights):
    avg_weight = tf.reduce_mean(weights_list_tuple, axis=0)
    avg_weights.append(avg_weight)

In [20]:

cnn_gru_mdpi = load_model('D:/linthoi/FL_models/new/glob_models/Type_glob_model/firstinitialized/CNN_GRU_MDPI_Type_glob_model.h5')
cnn_gru_mdpi.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn_gru_mdpi.set_weights(avg_weights)
cnn_gru_mdpi.save('D:/linthoi/FL_models/new/glob_models/Type_glob_model/updated_models/Final_CNN_GRU_MDPI_type_model.h5')

Validation of the global models in the FL_mu_dataset

In [2]:
fl_mu = pd.read_csv(r'D:\linthoi\FL_MU.csv')

C:\Users\ranja\AppData\Local\Temp\ipykernel_19356\576184422.py:1: DtypeWarning: Columns (87) have mixed types. Specify dtype option on import or set low_memory=False.
  fl_mu = pd.read_csv(r'D:\linthoi\FL_MU.csv')


In [5]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

In [6]:
fl_mu['label'] = label_encoder.fit_transform(fl_mu['label'])
category_mapping = pd.DataFrame({
    'label': label_encoder.classes_,
    'Encoded Value': label_encoder.transform(label_encoder.classes_)
})

# Display the result
print("Original label and Encoded Values:")
print(category_mapping)


fl_mu['category'] = label_encoder.fit_transform(fl_mu['category'])
category_mapping = pd.DataFrame({
    'category': label_encoder.classes_,
    'Encoded Value': label_encoder.transform(label_encoder.classes_)
})

# Display the result
print("Original Categories and Encoded Values:")
print(category_mapping)

fl_mu['type'] = label_encoder.fit_transform(fl_mu['type'])
category_mapping = pd.DataFrame({
    'type': label_encoder.classes_,
    'Encoded Value': label_encoder.transform(label_encoder.classes_)
})

# Display the result
print("Original Categories and Encoded Values:")
print(category_mapping)

Original label and Encoded Values:
    label  Encoded Value
0  Normal              0
1  attack              1
Original Categories and Encoded Values:
               category  Encoded Value
0                  DDOS              0
1                   DoS              1
2                  MiTM              2
3                Normal              3
4  Selective_Forwarding              4
5              Sinkhole              5
6                 Sybil              6
Original Categories and Encoded Values:
                    type  Encoded Value
0             AMQP_flood              0
1          ARP_Poisoning              1
2           DNS_Sinkhole              2
3      DNS_amplification              3
4     MQTT_connect_flood              4
5                 Normal              5
6             SOAP_Flood              6
7            STOMP_Flood              7
8   Selective_Forwarding              8
9                Slowite              9
10                 Sybil             10
11         TCP_SYN

In [11]:
fl_mu = fl_mu.dropna(axis=1, how='any')

In [12]:
fl_mu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18792330 entries, 0 to 18792329
Data columns (total 99 columns):
 #   Column       Dtype  
---  ------       -----  
 0   FPT          float64
 1   LPT          float64
 2   FD           float64
 3   FAMin        float64
 4   FAMean       float64
 5   FAMax        float64
 6   FAStd        float64
 7   FIMean       float64
 8   FIMin        float64
 9   FIMax        float64
 10  FIStd        float64
 11  DSCP_C       int64  
 12  DSCP_M       int64  
 13  DSCP_UV      int64  
 14  DSCP_S       int64  
 15  ECN_C        int64  
 16  ECN_M        int64  
 17  ECN_UV       int64  
 18  ECN_S        int64  
 19  FPC          int64  
 20  TPackets     int64  
 21  TBytes       int64  
 22  SBytes       int64  
 23  DBytes       int64  
 24  PLM          int64  
 25  FI           float64
 26  FIAT         float64
 27  BIAT         int64  
 28  ForwardPC    int64  
 29  BackwardPC   int64  
 30  FlowR        float64
 31  FlowStd      float64
 

In [13]:
col_remove = ['SIP', 'DIP', 'SPort', 'DPort', 'SIT', 'STT']
fl_mu=fl_mu.drop(col_remove, axis=1, inplace=False)
irr_list = []
for i in fl_mu:
    if len(fl_mu[i].unique())<2:
        irr_list.append(i)
fl_mu=fl_mu.drop(irr_list, axis=1, inplace=False)

In [14]:
X1=fl_mu.iloc[:, :-3]
y1 = fl_mu.iloc[:, -1]
X1 = scaler.fit_transform(X1)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.3, random_state=42)
X1_train_r = X1_train.reshape((X1_train.shape[0], 1 , X1_train.shape[1]))
X1_test_r = X1_test.reshape((X1_test.shape[0], 1 ,X1_test.shape[1]))

In [3]:
cnn_gru_type = load_model('D:/linthoi/FL_models/new/glob_models/Type_glob_model/updated_models/Final_CNN_GRU_MDPI_type_model.h5')

In [4]:
cnn_gru_type.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [29]:
# from tensorflow.keras.layers import Dense
# from tensorflow.keras.models import Sequential

# new_model = Sequential()

# # Add the layers from the original model, except the last (output) layer
# for layer in cnn_gru_type.layers[:-1]:
#     new_model.add(layer)

# # Add the new output layer with the desired number of neurons (e.g., 11 for multi-class classification)
# new_model.add(Dense(11, activation='softmax'))  # Adjust '11' to your new output size

# # Optionally, freeze the layers of the original model to keep the weights intact
# for layer in new_model.layers[:-1]:  # Freeze all layers except the output layer
#     layer.trainable = False

# # Compile the new model
# new_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [30]:
# new_model.summary()

In [27]:
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor = 'val_loss', patience = 5, verbose = 1, restore_best_weights=True)

# checkpoint_callback = ModelCheckpoint('D:/linthoi/FL_models/new/FL_mu_cnn_gr_type_weight.keras', monitor='val_loss', save_best_only=True, mode='min', verbose=1)
# new_model.fit(X1_train_r, y1_train, epochs=10, batch_size=32, validation_split=0.2, callbacks=[checkpoint_callback, early_stopping])
# y1_pred = new_model.predict(X1_test_r) 
# y1_pred_labels = np.argmax(y1_pred, axis=1)
# accuracy = accuracy_score(y1_test, y1_pred_labels)
# print(f"Accuracy Score: {accuracy}")

# # Generate the confusion matrix

# conf_matrix = confusion_matrix(y1_test, y1_pred_labels)

# # Visualize the confusion matrix using a heatmap
# plt.figure(figsize=(8, 6))
# sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=np.unique(y1_test), yticklabels=np.unique(y1_test))
# plt.xlabel('Predicted Labels')
# plt.ylabel('True Labels')
# plt.title('Confusion Matrix')
# plt.show()
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
# accuracy = accuracy_score(y1_test, y1_pred_labels)

# # Compute precision
# precision = precision_score(y1_test, y1_pred_labels, average='macro')

# # Compute recall
# recall = recall_score(y1_test, y1_pred_labels, average='macro')

# # Compute F1 score
# f1 = f1_score(y1_test, y1_pred_labels, average='macro')

# # Compute Matthews correlation coefficient (MCC)
# mcc = matthews_corrcoef(y1_test, y1_pred_labels)


In [28]:
# print(f"Accuracy for 5 class classification cnn_gru_type_final: {accuracy:.4f}")
# print(f"Precision for 4 class classification cnn_gru_type_final: {precision:.4f}")
# print(f"Recall for 4 class classification cnn_gru_type_final: {recall:.4f}")
# print(f"F1 Score for 4 class classification cnn_gru_type_final: {f1:.4f}")
# print(f"MCC for 4 class classification cnn_gru_type_final: {mcc:.4f}")